In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import DataLoader
import torch.utils.data as data
import torch.nn as nn
from sklearn.preprocessing import RobustScaler
import sys
from itertools import product

!rm -r /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

rm: cannot remove '/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory
Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 240, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 240 (delta 14), reused 25 (delta 8), pack-reused 207 (from 1)
Receiving objects: 100% (240/240), 2.82 MiB | 18.27 MiB/s, done.
Resolving deltas: 100% (125/125), done.
578832e (HEAD -> main, origin/main, origin/HEAD) forgot [rint condition


In [ ]:
sys.path.append("/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection")
import dataset
from lstm import LSTMBaseline
import train
if torch.cuda.is_available():
    device = torch.device("cuda")

### Set random states for reproducibility

In [3]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Create train and test datasets

### Phase One: small datasets for broad sweep over hidden_size, and num_layers with rest fixed

In [ ]:
train_dataset = dataset.forecasting_Dataset(device, 200, 4, start = 0, end = 90000)
test_dataset = dataset.forecasting_Dataset(device, 200, 4, train_dataset.scaler, start = 90000, end = 100000, train = False)
fixed_grid = {
    'horizon': 4,
    'lr': 0.001,
    'batch_size': 128
}
phase_one_grid = {
    "hidden_size": [64, 128, 256],
    "num_layers": [1, 2]    
}

In [ ]:
p1_results = []
for config in product(*phase_one_grid.values()):
    config_dict = dict(zip(phase_one_grid.keys(), config))
    name = f"p1_h{config_dict['hidden_size']}_nl{config_dict['num_layers']}"
    model = LSTMBaseline(config_dict["hidden_size"], fixed_grid["horizon"], config_dict["num_layers"]).to(device)
    train_dataset.l, test_dataset.l = fixed_grid["horizon"], fixed_grid["horizon"]
    weights, val_loss, train_loss = \
    train.fit_forecaster(device, model, name, train_dataset, test_dataset, fixed_grid['lr'], fixed_grid['batch_size'], 50)
    p1_results.append({**config_dict, **fixed_grid, 'val_loss': val_loss, 'name': name})
p1_results.sort(key=lambda x: x['val_loss'])
p1_top2 = p1_results[:2]
print("P1 survivors:", [(r['name'], r['val_loss']) for r in p1_top2])

|p1_h64_nl1| train = 0.0258, test= 0.0494
update LR: 0.001 -> 0.0005
|p1_h64_nl1| train = 0.0242, test= 0.0373
|p1_h64_nl1| train = 0.0239, test= 0.0347
update LR: 0.0005 -> 0.00025
|p1_h64_nl1| train = 0.0236, test= 0.0323
update LR: 0.00025 -> 0.000125
update LR: 0.000125 -> 6.25e-05
|p1_h64_nl1| train = 0.0234, test= 0.0319
|p1_h64_nl2| train = 0.0315, test= 0.1418
update LR: 0.001 -> 0.0005
|p1_h64_nl2| train = 0.0256, test= 0.0647
update LR: 0.0005 -> 0.00025
|p1_h64_nl2| train = 0.0244, test= 0.0540
update LR: 0.00025 -> 0.000125
|p1_h64_nl2| train = 0.0239, test= 0.0525
update LR: 0.000125 -> 6.25e-05
update LR: 6.25e-05 -> 3.125e-05
|p1_h64_nl2| train = 0.0236, test= 0.0460
|p1_h128_nl1| train = 0.0264, test= 0.0550
|p1_h128_nl1| train = 0.0248, test= 0.0424
update LR: 0.001 -> 0.0005
|p1_h128_nl1| train = 0.0236, test= 0.0355
update LR: 0.0005 -> 0.00025
|p1_h128_nl1| train = 0.0232, test= 0.0332
update LR: 0.00025 -> 0.000125
|p1_h128_nl1| train = 0.0229, test= 0.0324
|p1_h12

### Phase Two: larger dataset for lookback_windows and learning rates.

In [ ]:
train_dataset = dataset.forecasting_Dataset(device, 100, 4, start = 0, end = 300000)
test_dataset = dataset.forecasting_Dataset(device, 100, 4, train_dataset.scaler, start = 300000, end = 350000, train = False)
phase_two_grid = {
    'l': [4, 8, 16],
    'lr': [0.001, 0.0005, 0.0001]  
}

In [ ]:
p2_results = []
for p1_config in p1_top2: 
    for config in product(*phase_two_grid.values()):
        config_dict = dict(zip(phase_two_grid.keys(), config))
        name = (f"p2"
        f"_h{p1_config['hidden_size']}"
        f"_nl{p1_config['num_layers']}"
        f"_horizon{config_dict['horizon']}"
        f"_lr{config_dict['lr']}")
        
        model = LSTMBaseline(p1_config["hidden_size"], config_dict["horizon"], p1_config["num_layers"]).to(device)
        train_dataset.l, test_dataset.l = config_dict["horizon"], config_dict["horizon"]
        weights, val_loss, train_loss = \
        train.fit_forecaster(device, model, name, train_dataset, test_dataset, config_dict['lr'], fixed_grid['batch_size'], 50)
        p2_results.append({
            'hidden_size': p1_config['hidden_size'],
            'num_layers':  p1_config['num_layers'],
            **config_dict,
            'val_loss': val_loss,
            'name': name
        })
p2_results.sort(key=lambda x: x['val_loss'])
p2_top2 = p2_results[:2]
print("P2 survivors:", [(r['name'], r['val_loss']) for r in p2_top2])

|p2_h64_nl1_l4_lr0.001| train = 0.0246, test= 0.0290
|p2_h64_nl1_l4_lr0.001| train = 0.0241, test= 0.0263
update LR: 0.001 -> 0.0005
|p2_h64_nl1_l4_lr0.001| train = 0.0233, test= 0.0240
update LR: 0.0005 -> 0.00025
|p2_h64_nl1_l4_lr0.001| train = 0.0228, test= 0.0228
update LR: 0.00025 -> 0.000125
update LR: 0.000125 -> 6.25e-05
|p2_h64_nl1_l4_lr0.001| train = 0.0226, test= 0.0222
|p2_h64_nl1_l4_lr0.0005| train = 0.0243, test= 0.0265
|p2_h64_nl1_l4_lr0.0005| train = 0.0237, test= 0.0243
|p2_h64_nl1_l4_lr0.0005| train = 0.0233, test= 0.0237
update LR: 0.0005 -> 0.00025
|p2_h64_nl1_l4_lr0.0005| train = 0.0229, test= 0.0229
update LR: 0.00025 -> 0.000125
update LR: 0.000125 -> 6.25e-05
|p2_h64_nl1_l4_lr0.0005| train = 0.0227, test= 0.0223
|p2_h64_nl1_l4_lr0.0001| train = 0.0247, test= 0.0282
|p2_h64_nl1_l4_lr0.0001| train = 0.0239, test= 0.0257
|p2_h64_nl1_l4_lr0.0001| train = 0.0237, test= 0.0246
|p2_h64_nl1_l4_lr0.0001| train = 0.0236, test= 0.0240
update LR: 0.0001 -> 5e-05
|p2_h64_nl1

### Phase Three: full training dataset

In [ ]:
train_dataset = dataset.forecasting_Dataset(device, 100, 10, start = 0, end = 800000)
test_dataset = dataset.forecasting_Dataset(device, 100, 10, train_dataset.scaler, start = 800000, end = 1000000, train = False)

In [ ]:
p3_results = []
for p2_config in p2_top2:
    name = (f"p3"
        f"_h{p2_config['hidden_size']}"
        f"_nl{p2_config['num_layers']}"
        f"_horizon{p2_config['horizon']}"
        f"_lr{p2_config['lr']}")
    model = LSTMBaseline(p2_config["hidden_size"], p2_config["horizon"], p2_config["num_layers"]).to(device)
    train_dataset.l, test_dataset.l = p2_config['horizon'], p2_config['horizon']
    weights, val_loss, train_loss = \
    train.fit_forecaster(device, model, name, train_dataset, test_dataset, p2_config['lr'], fixed_grid['batch_size'], 50)
    p3_results.append({**p2_config, "weights": weights, 'val_loss': val_loss, "train_loss": train_loss, 'name': name})

p3_results.sort(key=lambda x: x['val_loss'])
best = p3_results[0]
print("Best model:", best['name'], "| val_loss:", best['val_loss'])

|p3_h64_nl1_l4_lr0.001| train = 0.0239, test= 0.0304
|p3_h64_nl1_l4_lr0.001| train = 0.0232, test= 0.0284
|p3_h64_nl1_l4_lr0.001| train = 0.0229, test= 0.0275
update LR: 0.001 -> 0.0005
|p3_h64_nl1_l4_lr0.001| train = 0.0222, test= 0.0235
update LR: 0.0005 -> 0.00025
|p3_h64_nl1_l4_lr0.001| train = 0.0220, test= 0.0224
update LR: 0.00025 -> 0.000125
|p3_h64_nl1_l4_lr0.0005| train = 0.0237, test= 0.0268
|p3_h64_nl1_l4_lr0.0005| train = 0.0233, test= 0.0253
update LR: 0.0005 -> 0.00025
|p3_h64_nl1_l4_lr0.0005| train = 0.0228, test= 0.0232
update LR: 0.00025 -> 0.000125
update LR: 0.000125 -> 6.25e-05
|p3_h64_nl1_l4_lr0.0005| train = 0.0221, test= 0.0227
update LR: 6.25e-05 -> 3.125e-05
update LR: 3.125e-05 -> 1.5625e-05
|p3_h64_nl1_l4_lr0.0005| train = 0.0220, test= 0.0219
Best model: p3_h64_nl1_l4_lr0.0005 | val_loss: 0.021881378205924173


### Save best

In [10]:
torch.save(best["weights"], 'best_LSTM_PURE_FORECASTER_weights.pth')